In [15]:
import os
os.chdir("/Users/caiwansun/Downloads/timeline-example-caiwan")
os.getcwd()

'/Users/caiwansun/Downloads/timeline-example-caiwan'

In [17]:
with open("samples-data/data-final/d2g2-segments-messi-gpt4-enact.json") as f:
    data = json.load(f)

print(type(data))
print(data)

<class 'dict'>
{'type': 'text', 'id': 'd2g2-segments-messi-gpt4-enact', 'algorithm': 'LLM', 'processed': 1772484241642, 'version': 1, 'localisation': [{'sublocalisations': {'localisation': [{'tclevel': 1, 'tcin': '00:00:07.0000', 'tcout': '00:00:19.0000', 'label': 'ENACTING MESSI'}, {'tclevel': 1, 'tcin': '00:00:07.0000', 'tcout': '00:00:23.0000', 'label': 'ENACTING MESSI'}, {'tclevel': 1, 'tcin': '00:00:07.0000', 'tcout': '00:00:23.0000', 'label': 'ENACTING MESSI'}, {'tclevel': 1, 'tcin': '00:00:07.0000', 'tcout': '00:00:31.0000', 'label': 'ENACTING MESSI'}, {'tclevel': 1, 'tcin': '00:00:07.0000', 'tcout': '00:00:31.0000', 'label': 'ENACTING MESSI'}, {'tclevel': 1, 'tcin': '00:00:07.0000', 'tcout': '00:00:46.0000', 'label': 'ENACTING MESSI'}, {'tclevel': 1, 'tcin': '00:00:07.0000', 'tcout': '00:00:46.0000', 'label': 'ENACTING MESSI'}, {'tclevel': 1, 'tcin': '00:00:07.0000', 'tcout': '00:00:51.0000', 'label': 'ENACTING MESSI'}, {'tclevel': 1, 'tcin': '00:00:15.0000', 'tcout': '00:00:38

In [19]:
# Method1: All pairs comparison with condition classification

import json

# classify temporal relation between two segments
def classify(a_start, a_end, b_start, b_end):

    if a_start == b_start and a_end == b_end:
        return "EXACT_MATCH"

    if a_start == b_start:
        return "SAME_START"

    if a_end == b_end:
        return "SAME_END"

    if b_start <= a_start and b_end >= a_end:
        return "GPT52_CONTAINS_GPT4"

    if a_start <= b_start and a_end >= b_end:
        return "GPT4_CONTAINS_GPT52"

    if a_start < b_end and a_end > b_start:
        return "PARTIAL_OVERLAP"

    return "NO_OVERLAP"


# convert HH:MM:SS → seconds
def tc_to_seconds(tc):
    h, m, s = tc.split(":")
    return int(h)*3600 + int(m)*60 + float(s)


# extract condition from justification text
def extract_condition(text):

    text = text.lower()

    if "condition 1" in text:
        return "condition_1"

    if "condition 2" in text:
        return "condition_2"

    if "condition 3" in text:
        return "condition_3"

    return "unknown"


# load original GPT segmentation outputs
with open("gpt4.json") as f:
    data4 = json.load(f)

gpt4 = data4["segments"]

with open("gpt52.json") as f:
    data52 = json.load(f)

gpt52 = data52["segments"]

print("GPT4 segments:", len(gpt4))
print("GPT52 segments:", len(gpt52))
print(gpt4[0])

GPT4 segments: 44
GPT52 segments: 189
{'justification': "The student's (Rose's) [gaze] is fixated on the [Screen] from 00:00:07 to 00:04:38. From time 00:00:15 to 00:00:38, the student's [movement] is equal to [moving]. Condition 1 states that if the student is moving and looking at the screen, then the student is ENACTING. Condition 1 is met on the intersection of Rose's gaze and movement from 00:00:15 to 00:00:38. By definition, this is an ENACTING segment.", 'time_in': '00:00:15', 'time_out': '00:00:38', 'label': 'ENACTING'}


In [21]:
# run all-pairs comparison with condition labels

results = []

for seg4 in gpt4:

    a_start = tc_to_seconds(seg4["time_in"])
    a_end = tc_to_seconds(seg4["time_out"])
    cond4 = extract_condition(seg4["justification"])

    for seg52 in gpt52:

        b_start = tc_to_seconds(seg52["time_in"])
        b_end = tc_to_seconds(seg52["time_out"])
        cond52 = extract_condition(seg52["justification"])

        relation = classify(a_start, a_end, b_start, b_end)

        results.append({
            "gpt4": [a_start, a_end],
            "gpt52": [b_start, b_end],
            "relation": relation,
            "condition4": cond4,
            "condition52": cond52
        })

print("Total comparisons:", len(results))

for r in results[:20]:
    print(r)

Total comparisons: 8316
{'gpt4': [15.0, 38.0], 'gpt52': [7.0, 9.0], 'relation': 'NO_OVERLAP', 'condition4': 'condition_1', 'condition52': 'condition_1'}
{'gpt4': [15.0, 38.0], 'gpt52': [15.0, 38.0], 'relation': 'EXACT_MATCH', 'condition4': 'condition_1', 'condition52': 'condition_1'}
{'gpt4': [15.0, 38.0], 'gpt52': [42.0, 61.0], 'relation': 'NO_OVERLAP', 'condition4': 'condition_1', 'condition52': 'condition_1'}
{'gpt4': [15.0, 38.0], 'gpt52': [73.0, 75.0], 'relation': 'NO_OVERLAP', 'condition4': 'condition_1', 'condition52': 'condition_1'}
{'gpt4': [15.0, 38.0], 'gpt52': [84.0, 162.0], 'relation': 'NO_OVERLAP', 'condition4': 'condition_1', 'condition52': 'condition_1'}
{'gpt4': [15.0, 38.0], 'gpt52': [166.0, 188.0], 'relation': 'NO_OVERLAP', 'condition4': 'condition_1', 'condition52': 'condition_1'}
{'gpt4': [15.0, 38.0], 'gpt52': [201.0, 219.0], 'relation': 'NO_OVERLAP', 'condition4': 'condition_1', 'condition52': 'condition_1'}
{'gpt4': [15.0, 38.0], 'gpt52': [224.0, 252.0], 'relati

In [27]:
from collections import defaultdict

# group statistics by relation first
relation_summary = defaultdict(lambda: defaultdict(int))

for r in results:

    relation = r["relation"]
    cond_pair = (r["condition4"], r["condition52"])

    relation_summary[relation][cond_pair] += 1


# print organized results
for relation, cond_counts in relation_summary.items():

    print("\nRelation:", relation)

    for cond_pair, count in cond_counts.items():
        print("   ", cond_pair, ":", count)


Relation: NO_OVERLAP
    ('condition_1', 'condition_1') : 375
    ('condition_1', 'condition_2') : 1222
    ('condition_1', 'condition_3') : 602
    ('condition_2', 'condition_1') : 169
    ('condition_2', 'condition_2') : 506
    ('condition_2', 'condition_3') : 263
    ('condition_3', 'condition_1') : 465
    ('condition_3', 'condition_2') : 1447
    ('condition_3', 'condition_3') : 689

Relation: EXACT_MATCH
    ('condition_1', 'condition_1') : 14
    ('condition_2', 'condition_2') : 7
    ('condition_3', 'condition_3') : 22
    ('condition_3', 'condition_2') : 14
    ('condition_2', 'condition_3') : 6

Relation: PARTIAL_OVERLAP
    ('condition_1', 'condition_2') : 39
    ('condition_1', 'condition_3') : 20
    ('condition_2', 'condition_1') : 7
    ('condition_3', 'condition_1') : 14

Relation: GPT52_CONTAINS_GPT4
    ('condition_1', 'condition_2') : 366
    ('condition_1', 'condition_3') : 186

Relation: SAME_START
    ('condition_2', 'condition_1') : 5
    ('condition_2', 'condi

In [25]:
# Method2: One-to-Many Overlap Mapping with condition labels

results_method2 = []

for i, seg4 in enumerate(gpt4):

    a_start = tc_to_seconds(seg4["time_in"])
    a_end = tc_to_seconds(seg4["time_out"])
    cond4 = extract_condition(seg4["justification"])

    related_segments = []

    for j, seg52 in enumerate(gpt52):

        b_start = tc_to_seconds(seg52["time_in"])
        b_end = tc_to_seconds(seg52["time_out"])
        cond52 = extract_condition(seg52["justification"])

        relation = classify(a_start, a_end, b_start, b_end)

        if relation != "NO_OVERLAP":

            related_segments.append({
                "GPT52_id": j+1,
                "segment": [b_start, b_end],
                "relation": relation,
                "condition52": cond52
            })

    results_method2.append({
        "GPT4_id": i+1,
        "GPT4_segment": [a_start, a_end],
        "condition4": cond4,
        "related_GPT52": related_segments
    })


# print examples
for r in results_method2[:3]:

    print("GPT4_", r["GPT4_id"], r["GPT4_segment"], r["condition4"])

    for seg in r["related_GPT52"]:
        print("   → GPT52_", seg["GPT52_id"], seg["segment"], seg["relation"], seg["condition52"])

GPT4_ 1 [15.0, 38.0] condition_1
   → GPT52_ 2 [15.0, 38.0] EXACT_MATCH condition_1
   → GPT52_ 27 [7.0, 19.0] PARTIAL_OVERLAP condition_2
   → GPT52_ 28 [7.0, 23.0] PARTIAL_OVERLAP condition_3
   → GPT52_ 29 [7.0, 23.0] PARTIAL_OVERLAP condition_2
   → GPT52_ 30 [7.0, 31.0] PARTIAL_OVERLAP condition_3
   → GPT52_ 31 [7.0, 31.0] PARTIAL_OVERLAP condition_2
   → GPT52_ 32 [7.0, 34.0] PARTIAL_OVERLAP condition_2
   → GPT52_ 33 [7.0, 41.0] GPT52_CONTAINS_GPT4 condition_2
   → GPT52_ 34 [7.0, 46.0] GPT52_CONTAINS_GPT4 condition_3
   → GPT52_ 35 [7.0, 46.0] GPT52_CONTAINS_GPT4 condition_2
   → GPT52_ 36 [7.0, 48.0] GPT52_CONTAINS_GPT4 condition_2
   → GPT52_ 37 [7.0, 51.0] GPT52_CONTAINS_GPT4 condition_3
   → GPT52_ 38 [7.0, 71.0] GPT52_CONTAINS_GPT4 condition_3
   → GPT52_ 39 [7.0, 90.0] GPT52_CONTAINS_GPT4 condition_2
   → GPT52_ 40 [7.0, 91.0] GPT52_CONTAINS_GPT4 condition_2
   → GPT52_ 41 [7.0, 93.0] GPT52_CONTAINS_GPT4 condition_2
   → GPT52_ 42 [7.0, 96.0] GPT52_CONTAINS_GPT4 conditio